# Chapter 3: Planes, Not Axes

## Bivectors as oriented planes, simple vs compound bivectors, principal plane extraction

In $\mathbb{R}^3$ you learned that every rotation has an **axis** — a single vector around which everything spins. This is a beautiful and useful fact, but it is a *low-dimensional accident*. In four or more dimensions, there is no unique rotation axis. A rotation in $\mathbb{R}^4$ can spin in *two independent planes simultaneously*, and there is no single vector that stays fixed.

The correct geometric object that describes a rotation is not a vector (axis) but a **plane** — the plane in which the rotation happens. In Geometric Algebra, the algebraic object that represents an oriented plane is the **bivector**.

$$B = \mathbf{a} \wedge \mathbf{b}$$

The wedge product $\wedge$ takes two vectors and produces a bivector — a directed area element that encodes:
- **which plane** the rotation lives in (spanned by $\mathbf{a}$ and $\mathbf{b}$),
- **how much** rotation (the magnitude $\|B\|$), and
- **which direction** (orientation, the sign of $B$).

This chapter teaches you to work with bivectors: create them, decompose compound bivectors into their constituent planes, and extract the principal rotation planes from real layer-to-layer transformations inside a transformer.

**By the end of this chapter you will be able to:**
- Construct simple and compound bivectors from vectors
- Decompose a general bivector into principal planes via eigenanalysis
- Read the rotation-plane spectrum of an actual transformer layer

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
from scipy.linalg import logm, expm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from layer_time_ga.algebra import (
    Bivector, bivector_from_skew, rotor_from_orthogonal,
    geometric_product_vectors
)

print("Imports OK")
print(f"NumPy {np.__version__}")

## Simple Bivectors: A Single Plane

The **wedge product** of two vectors $\mathbf{a}$ and $\mathbf{b}$ produces a **simple bivector**:

$$B = \mathbf{a} \wedge \mathbf{b}$$

A simple bivector represents *exactly one plane* of rotation. In our matrix representation, the bivector is a **skew-symmetric matrix**:

$$B_{ij} = a_i b_j - a_j b_i$$

which is just `np.outer(a, b) - np.outer(b, a)`.

Key properties of a simple bivector:
- It has **rank 2** as a matrix (only two nonzero eigenvalues, forming a conjugate pair $\pm i\sigma$).
- The eigenvalue magnitude $\sigma$ gives the **rotation angle**.
- The eigenvectors span the **plane of rotation**.

Let us build one from two orthonormal vectors and verify it has exactly one principal plane.

In [ ]:
# ------------------------------------------------------------------
# Simple bivector from two orthonormal vectors in R^6
# ------------------------------------------------------------------
dim = 6

# Two orthonormal vectors: e_0 and e_1
a = np.zeros(dim); a[0] = 1.0   # e_0
b = np.zeros(dim); b[1] = 1.0   # e_1

# Wedge product: a ^ b  (skew-symmetric matrix)
wedge_matrix = np.outer(a, b) - np.outer(b, a)
print("Wedge product matrix (a ^ b):")
print(wedge_matrix)
print()

# Wrap as a Bivector object
B_simple = bivector_from_skew(wedge_matrix)

print(f"Bivector dimension: {B_simple.dim}")
print(f"Bivector norm ||B||_F: {B_simple.norm:.4f}")
print(f"Rotation angle: {B_simple.angle:.4f} rad")
print()

# Extract principal planes — should find exactly ONE nonzero plane
planes = B_simple.principal_planes(n_planes=5)
print(f"Number of principal planes found: {len(planes)}")
for i, p in enumerate(planes):
    print(f"  Plane {i}: angle = {p['angle']:.4f}, weight = {p['weight']:.4f}")

print("\n=> A simple bivector has exactly ONE principal plane.")

## Compound Bivectors: Multiple Planes

A **simple** bivector is a single wedge product $\mathbf{a} \wedge \mathbf{b}$ — it represents rotation in exactly one plane. But a **general** bivector is a *sum* of simple bivectors:

$$B = \sigma_1 (\mathbf{u}_1 \wedge \mathbf{v}_1) + \sigma_2 (\mathbf{u}_2 \wedge \mathbf{v}_2) + \cdots$$

This represents **simultaneous rotation in multiple orthogonal planes**, each with its own magnitude $\sigma_j$. This is exactly what happens in high-dimensional spaces like a transformer's hidden-state space: a single layer transition rotates the representation in *many planes at once*.

As a skew-symmetric matrix, a compound bivector has **multiple conjugate eigenvalue pairs** $\pm i\sigma_j$, each corresponding to an independent plane of rotation. The magnitudes $\sigma_j$ tell you how much rotation happens in each plane.

Let us build a compound bivector by adding two simple bivectors in orthogonal planes.

In [ ]:
# ------------------------------------------------------------------
# Compound bivector: two simple bivectors in orthogonal planes
# ------------------------------------------------------------------
dim = 6

# Plane 1: e_0 ^ e_1, with weight sigma_1 = 0.8
e0 = np.zeros(dim); e0[0] = 1.0
e1 = np.zeros(dim); e1[1] = 1.0
B1_matrix = 0.8 * (np.outer(e0, e1) - np.outer(e1, e0))
B1 = bivector_from_skew(B1_matrix)

# Plane 2: e_2 ^ e_3, with weight sigma_2 = 0.3
e2 = np.zeros(dim); e2[2] = 1.0
e3 = np.zeros(dim); e3[3] = 1.0
B2_matrix = 0.3 * (np.outer(e2, e3) - np.outer(e3, e2))
B2 = bivector_from_skew(B2_matrix)

# Compound bivector: B = B1 + B2
B_compound = B1 + B2

print("Compound bivector matrix (6x6):")
print(np.round(B_compound.matrix, 3))
print()
print(f"Compound bivector norm: {B_compound.norm:.4f}")
print(f"Aggregate rotation angle: {B_compound.angle:.4f} rad")
print()

# Extract principal planes — should find exactly TWO
planes = B_compound.principal_planes(n_planes=5)
print(f"Number of principal planes found: {len(planes)}")
for i, p in enumerate(planes):
    print(f"  Plane {i}: weight = {p['weight']:.4f} (expected: {[0.8, 0.3][i] if i < 2 else '0.0'})")

print("\n=> The compound bivector decomposes into exactly two planes,")
print("   with weights matching our construction (0.8 and 0.3).")

## Principal Plane Extraction

Any skew-symmetric matrix $B$ can be decomposed into its **principal planes** via eigendecomposition. Since $B$ is real and skew-symmetric, its eigenvalues come in conjugate pairs $\pm i\sigma_j$ (plus zeros for any unrotated dimensions). Each pair defines:

1. **A simple bivector** (a single plane of rotation), spanned by the real and imaginary parts of the eigenvector.
2. **A rotation magnitude** $\sigma_j$, telling you how much rotation happens in that plane.

Sorting by $\sigma_j$ descending gives the **principal plane spectrum** — the rotation analog of the singular value spectrum. The first principal plane captures the largest rotation; subsequent planes capture progressively smaller rotations.

This is exactly the tool we need for transformer analysis. A layer's bivector generator lives in $\mathbb{R}^{256 \times 256}$ and can rotate in up to 128 planes simultaneously. The principal plane decomposition tells us which planes carry the most rotation and how the rotation "budget" is distributed.

Let us try this on a random 8-dimensional skew-symmetric matrix.

In [ ]:
# ------------------------------------------------------------------
# Principal plane decomposition of a random 8x8 skew-symmetric matrix
# ------------------------------------------------------------------
np.random.seed(42)
dim = 8

# Generate a random skew-symmetric matrix
A = np.random.randn(dim, dim)
A_skew = 0.5 * (A - A.T)  # enforce skew-symmetry

B_random = bivector_from_skew(A_skew)
print(f"Random bivector in R^{dim}")
print(f"  Norm: {B_random.norm:.4f}")
print(f"  Aggregate angle: {B_random.angle:.4f} rad")
print(f"  Max possible planes: {dim // 2}")
print()

# Extract all principal planes (dim/2 = 4 planes in R^8)
planes = B_random.principal_planes(n_planes=dim // 2)

print("Principal plane spectrum:")
print("-" * 40)
weights = []
for i, p in enumerate(planes):
    print(f"  Plane {i}: angle = {p['angle']:.4f} rad, weight = {p['weight']:.4f}")
    weights.append(p['weight'])

# Bar chart of principal plane weights
fig, ax = plt.subplots(figsize=(7, 4))
plane_labels = [f"Plane {i}" for i in range(len(weights))]
bars = ax.bar(plane_labels, weights, color=['#4477AA', '#EE6677', '#228833', '#CCBB44'])
ax.set_ylabel('Rotation magnitude $\\sigma_j$')
ax.set_title(f'Principal Plane Spectrum (random bivector in $\\mathbb{{R}}^{{{dim}}}$)')
ax.set_xlabel('Principal plane (sorted by weight)')

# Annotate bars with values
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{w:.3f}', ha='center', va='bottom', fontsize=10)

fig.tight_layout()
plt.savefig('ch03_principal_planes_random.png', dpi=150, bbox_inches='tight')
print("\nSaved: ch03_principal_planes_random.png")
plt.show()

## Bivector Arithmetic

Bivectors form a **vector space**: you can add them, subtract them, and scale them by real numbers. This is because the set of skew-symmetric $k \times k$ matrices is itself a vector space of dimension $k(k-1)/2$.

Concretely:
- **Addition** $B_1 + B_2$: the sum of two bivectors is another bivector (another skew-symmetric matrix).
- **Scalar multiplication** $\alpha B$: scales all rotation magnitudes uniformly.
- **Subtraction** $B_1 - B_2$: measures the "difference" between two rotations.

The **Frobenius norm** $\|B\|_F = \sqrt{\sum_{i<j} B_{ij}^2 \cdot 2}$ measures the total rotation magnitude. It satisfies the triangle inequality, so the distance between two bivectors $\|B_1 - B_2\|_F$ is a meaningful measure of how different the two rotations are.

This linearity is one reason the bivector representation is so powerful for analyzing transformers: we can compute averages, differences, and interpolations of rotation patterns across layers.

In [ ]:
# ------------------------------------------------------------------
# Bivector arithmetic: addition, subtraction, scalar multiplication
# ------------------------------------------------------------------
dim = 6

# Reuse B1 (plane e_0^e_1, weight 0.8) and B2 (plane e_2^e_3, weight 0.3)
print("B1: rotation in e_0 ^ e_1 plane, weight 0.8")
print(f"  ||B1|| = {B1.norm:.4f}")
print()

print("B2: rotation in e_2 ^ e_3 plane, weight 0.3")
print(f"  ||B2|| = {B2.norm:.4f}")
print()

# Addition
B_sum = B1 + B2
print(f"B1 + B2:")
print(f"  ||B1 + B2|| = {B_sum.norm:.4f}")
print(f"  Expected (Pythagorean, since orthogonal planes): "
      f"sqrt(||B1||^2 + ||B2||^2) = {np.sqrt(B1.norm**2 + B2.norm**2):.4f}")
print()

# Subtraction
B_diff = B1 - B2
print(f"B1 - B2:")
print(f"  ||B1 - B2|| = {B_diff.norm:.4f}")
print()

# Scalar multiplication
B_scaled = 2.0 * B1
print(f"2 * B1:")
print(f"  ||2*B1|| = {B_scaled.norm:.4f}")
print(f"  Expected: 2 * ||B1|| = {2 * B1.norm:.4f}")
print()

# Verify: scaling preserves plane structure
planes_scaled = B_scaled.principal_planes(n_planes=3)
print(f"Planes of 2*B1: {len(planes_scaled)} plane(s)")
print(f"  Weight of dominant plane: {planes_scaled[0]['weight']:.4f} (expected: 1.6)")

print("\n=> Bivectors support the standard vector space operations.")

## In the Transformer: Layer Rotation Planes

Now we apply everything to a real transformer. Each layer transition $l \to l+1$ is decomposed (via polar decomposition) into a **rotor** $R^{(l)}$ and a **metric deformation** $P^{(l)}$. The rotor's bivector generator $B^{(l)} = \log(R^{(l)})$ is the skew-symmetric matrix whose principal planes tell us *which subspaces* of the hidden-state space are being rotated at layer $l$.

We will:
1. Load a model and run a prompt through it.
2. Extract the bivector field (one bivector per layer transition).
3. For three representative layers (early, middle, late), extract the top principal planes.
4. Visualize the principal plane spectra as a grouped bar chart.

This reveals how the **rotation structure changes across the network**: early layers may rotate in a few dominant planes (coarse feature alignment), while later layers may spread rotation across many planes (fine-grained feature mixing).

In [ ]:
# ------------------------------------------------------------------
# Extract rotation planes from a real transformer
# ------------------------------------------------------------------
import ltg_ga

# Load model (this may take a moment on first run)
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Loaded: {model.name}")
print(f"  Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")

# Analyse a prompt
result = ltg_ga.analyse(
    "The Pythagorean theorem states that in a right triangle",
    model=model,
    compute_dependency=False  # skip dependency for speed
)
print(f"\nAnalysis complete: {result.n_layers} layers, k={result.k}")
print(f"Bivector field: {len(result.bivector_field)} bivectors")

In [ ]:
# ------------------------------------------------------------------
# Principal planes at early, middle, and late layers
# ------------------------------------------------------------------
n_biv = len(result.bivector_field)
n_top_planes = 5  # top-5 principal planes per layer

# Pick three representative layers: early, middle, late
layer_indices = {
    "Early (layer 2)": min(2, n_biv - 1),
    f"Middle (layer {n_biv // 2})": n_biv // 2,
    f"Late (layer {n_biv - 2})": n_biv - 2,
}

print("Principal plane spectra for three representative layers:")
print("=" * 60)

all_weights = {}
for label, idx in layer_indices.items():
    biv = result.bivector_field[idx]
    planes = biv.principal_planes(n_planes=n_top_planes)
    weights = [p['weight'] for p in planes]
    all_weights[label] = weights

    print(f"\n{label}:")
    print(f"  Total rotation angle: {biv.angle:.4f} rad")
    for j, p in enumerate(planes):
        print(f"    Plane {j}: sigma = {p['weight']:.4f}")

# ------------------------------------------------------------------
# Grouped bar chart
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(n_top_planes)
width = 0.25
colors = ['#4477AA', '#EE6677', '#228833']

for i, (label, weights) in enumerate(all_weights.items()):
    # Pad if fewer planes found
    w = weights + [0.0] * (n_top_planes - len(weights))
    ax.bar(x + i * width, w[:n_top_planes], width, label=label, color=colors[i])

ax.set_xlabel('Principal plane index (sorted by magnitude)')
ax.set_ylabel('Rotation magnitude $\\sigma_j$')
ax.set_title('Principal Rotation Planes: Early vs Middle vs Late Layers')
ax.set_xticks(x + width)
ax.set_xticklabels([f'Plane {j}' for j in range(n_top_planes)])
ax.legend()
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.savefig('ch03_layer_rotation_planes.png', dpi=150, bbox_inches='tight')
print("\nSaved: ch03_layer_rotation_planes.png")
plt.show()

## Exercises

1. **Simple bivector in R^10.** Choose two random (non-orthonormal) vectors in $\mathbb{R}^{10}$, compute their wedge product, and verify that the resulting bivector has exactly one principal plane. What is the relationship between the plane's weight and the area of the parallelogram spanned by the two vectors?

2. **Rank and plane count.** Create a compound bivector in $\mathbb{R}^8$ with exactly three nonzero principal planes (weights 1.0, 0.5, 0.2). Verify that the matrix rank is 6 (= 2 per plane). What happens to the rank when two of the planes have equal weight?

3. **Plane concentration ratio.** For the transformer analysis above, define the *concentration ratio* as $\sigma_1 / \sum_j \sigma_j$ — the fraction of total rotation captured by the dominant plane. Compute this for every layer. Do early or late layers have higher concentration? What might this mean for the type of information processing each layer does?

4. **Bivector distance.** Pick two layers whose rotor angles are similar but whose curvature contributions differ. Compute $\|B^{(l_1)} - B^{(l_2)}\|_F$. Is it possible for two layers to have the same rotation angle but very different bivectors? What would that mean geometrically?

5. **From rotor to bivector and back.** Take a rotor $R^{(l)}$ from the analysis, extract its bivector $B = \log(R)$, then reconstruct the rotor via $R' = \exp(B)$. Verify that $\|R - R'\|_F \approx 0$. Use `scipy.linalg.expm` and `logm`.